# Capabilities: choose a question, then a method

Onceover searches for several kinds of structure in the same table. A
finding's category tells you what it states; its evidence identifies the
method that found it. Different methods can support the same statement,
while a single method can produce several kinds of findings.

The standard budget runs the default detector set. You can focus the work
with `detectors=[...]`, or declare a numeric response with `target=`.
The `views` detector is experimental and opt-in. This guide starts with
user questions; the registry at the end gives the exact detector IDs.

## Can I use this table in a join?

[Keys and duplicates](../methodology/keys_duplicates.ipynb) distinguish a
verified single-column key, a composite key, a nearly unique column, and
repeated rows. These are different properties: a repeated key value need
not identify an exact duplicate row. Counts and final key/duplicate
comparisons cover the full input.

For an extract with repeated orders, mixed invoice formats, and missing
dates, follow the [vendor-extract case study](../case_studies/vendor_extract.ipynb).

## Are values missing or recorded inconsistently?

[Column cards](columns.ipynb) summarize types, nulls, exact distinct counts,
concentration, top values, and character-format patterns. Role suggestions
such as `identifier` or `categorical` are inferences from those counts.

[Sentinels](../methodology/sentinel_values.ipynb) identify repeated,
windowed values that could encode missingness. [Digit fingerprints](../methodology/digit_fingerprints.ipynb)
expose rounding and digit preferences. The `granularity` detector identifies
changes between fractional and integer-valued recording. These findings
point to a production process to inspect, not an automatic recoding rule.

[Extreme-value detection](../methodology/extreme_values.ipynb) finds numeric
values isolated from their column or a categorical subset, and categorical
values that break a nearly universal pattern inside a context. Its evidence
identifies the rows and the reference group.

## Which columns repeat or derive information?

[Functional dependencies](../methodology/functional_dependencies.ipynb)
check whether equal inputs map to equal outputs. Approximate mappings
report violations. Arithmetic scans verify affine formulas, sums,
differences, products, ratios, and small linear combinations on complete
rows, within a stated tolerance.

The `duplicate_encodings` detector identifies alternative labels for the
same grouping and finer-to-coarser rollups. The `pivot` detector recognizes
a verified long layout and profiles its wide form as well; wide-form
findings are tagged in their evidence. See
[long-format reports](findings.ipynb).

A [predictability finding](../methodology/predictability_screen.ipynb)
answers a different question: can a fitted model predict the target on
held-out rows? It can reveal nonlinear structure without establishing an
exact mapping. The [feature-trimming case study](../case_studies/feature_trimming.ipynb)
shows how to separate derivation, predictability, and a merely unique
continuous measurement.

## What happens within a segment?

[PRISM and decision tables](../methodology/rules_tables.ipynb) describe
conditional patterns and compact predictive tables.
[Association mining](../methodology/associations.ipynb) finds pairwise
nominal implications. Reported rules include a common diagnostic recount:
matching rows, success rate, baseline, and lift.

[Hotspots](../methodology/hotspots.ipynb) find combinations whose joint
frequency exceeds what the marginals explain. [Local linear regimes](../methodology/local_linear_regimes.ipynb)
fit different equations in different regions and compare them with a
single global line on held-out rows.

## What shape does a relationship have?

[Pearson, Spearman, and Chatterjee's ξ](../methodology/dependence_measures.ipynb)
describe linear, monotone, and more general pairwise dependence.
[Period associations](../methodology/period_associations.ipynb) can reveal
a sign change hidden by a pooled correlation.

When the response is known, [target shapes](../methodology/target_shapes.ipynb)
compare a line, a binned curve, and a single step. The
[interaction pass](../methodology/interactions.ipynb) asks whether two
predictors act non-additively in a fitted model of a target. An interaction
is not the same as association between the two predictors themselves.

The experimental [views detector](../methodology/view_clusters.ipynb)
asks which columns can share a fitted clustering of rows. Its
[method reference](../methodology/view_clusters_reference.ipynb) explains
the score, search, and additional cluster/density findings.

## Where does behavior change?

[Temporal analysis](../methodology/temporal_analysis.ipynb) covers inferred
calendar gaps, irregular intervals, ordering, appended backfills, sustained
level/trend/variance changes, and transient excursions. A
[residual shift](../methodology/local_linear_regimes.ipynb) measures change
after accounting for a stated reference model.

[Discontinuities](../methodology/step_discontinuities.ipynb) find threshold
jumps across numeric axes, including diagonal boundaries, without
requiring time order. [Drift](../methodology/drift.ipynb) compares how later
periods route through an early-period model. Read its scores as changes in
model-used predictor distributions, not a general test of target behavior.

[Declared expectations](../methodology/expectations.ipynb) check a supplied
calendar, complete sequence, lexicographic ordering, or panel layout.
They distinguish observed irregularity from a violation of a stated
expectation.

## Select the work explicitly

This small fixture has a nonlinear amount. A target-aware run adds a shape
description to the normal profile. An explicit detector list instead
replaces the default selection:


In [ ]:
import numpy as np
import polars as pl
import onceover

rng = np.random.default_rng(42)
x = rng.uniform(-1, 1, 800)
df = pl.DataFrame({"driver": x, "amount": 100 + 30 * x * x + rng.normal(0, 0.2, len(x))})

shaped = onceover.profile(df, target="amount")
for finding in shaped.select(category="dependencies", target="amount"):
    print(finding["specification"].get("form"), finding["claim"])

focused = onceover.profile(df, detectors=["numeric_dependence", "identities"])
print("Focused run:")
print(focused.to_digest(1_200))


With an explicit detector list, include `target_shapes` if you want shape
analysis. Required resource detectors can run without contributing their
own findings. `quick` omits model-based relationship searches; larger
budgets widen the methods' own limits. See [coverage](coverage.ipynb) and
[the API reference](api.ipynb) for exact controls.

## Detector registry

The following registry comes from the installed package. The purpose column
connects each ID to its role in the guide. A registered detector can still
be ineligible for a particular input, require a declared target/time axis,
or be omitted by the budget. Read its coverage rather than inferring work
from its presence in the registry.


In [ ]:
from great_tables import GT, loc, style

purposes = {
    "column_profile": "Column summaries, role suggestions, missingness, concentration, string formats",
    "row_duplicates": "Repeated rows under the stated comparison and missing-value policy",
    "keys": "Single-column, composite, and nearly unique keys",
    "pivot": "Verified long layout and relationships in its wide form",
    "functional_dependencies": "Exact or approximate value mappings, including multi-column determinants",
    "duplicate_encodings": "Alternative labels for a grouping and finer-to-coarser rollups",
    "sortedness": "Order in the row sequence and late appended backfills",
    "granularity": "Changes between fractional and integer-valued recording",
    "sentinels": "Repeated, windowed candidate missing-value codes",
    "digits": "Heaping, terminal-digit preferences, and eligible Benford deviations",
    "dates": "Inferred calendar gaps and interval irregularity",
    "expectations": "Declared sequences, calendars, ordering, and panel completeness",
    "gbm": "Predictability of eligible columns from the other columns",
    "decision_tables": "Compact predictive tables selected by cross-validation",
    "prism": "Multi-condition rules for categorical targets",
    "regimes": "Local linear equations and dated reference-model residual shifts",
    "interactions": "Non-additive predictor pairs under a fitted target model",
    "target_shapes": "Line, curve, and step comparisons for a designated numeric target",
    "breaks": "Sustained level, trend, variance changes, and transient excursions",
    "surprises": "Isolated numeric values and context-breaking categorical values",
    "discontinuities": "Axis-aligned and diagonal threshold jumps",
    "drift": "Reference-period model-routing changes along a declared time axis",
    "association_rules": "Pairwise nominal rules with train and diagnostic evidence",
    "hotspots": "Combinations concentrated beyond their marginal frequencies",
    "numeric_dependence": "Pearson, Spearman, and both directions of Chatterjee's xi",
    "identities": "Affine and small multi-term arithmetic relationships",
    "period_associations": "Changes in pairwise correlation across calendar or year periods",
    "views": "Experimental shared row partitions, cluster differences, and density findings",
}
registry = onceover.available_detectors()
assert set(purposes) == {d["id"] for d in registry}
registry_rows = [{"Detector": d["id"], "Purpose": purposes[d["id"]],
                  "Resource dependencies": ", ".join(d["requires"]) or "None"}
                 for d in registry]
(GT(pl.DataFrame(registry_rows))
 .tab_header(title="Available detectors", subtitle=f"Installed onceover {onceover.__version__}")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))
